In [ ]:
from google.colab import drive, userdata, output
import os
from pathlib import Path

drive.mount('/content/drive')
output.enable_custom_widget_manager()
DRIVE_ROOT = Path('/content/drive/MyDrive')
MODEL_ROOT = DRIVE_ROOT / 'models'
RESULT_ROOT = DRIVE_ROOT / 'cbt_results'
BASE_MODEL_DIR = MODEL_ROOT / 'Qwen2.5-32B-Instruct'
LORA_DIR = MODEL_ROOT / 'cbt-qwen32b-lora'
ROBERTA_DIR = MODEL_ROOT / 'roberta-large'
for p in [MODEL_ROOT, RESULT_ROOT, BASE_MODEL_DIR, LORA_DIR, ROBERTA_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print('Folders ready:', MODEL_ROOT, RESULT_ROOT, sep='\n')


Mounted at /content/drive
Folders ready:
/content/drive/MyDrive/models
/content/drive/MyDrive/cbt_results


## Optional cleanup
Only run if you want to remove old or incomplete downloads.


In [ ]:
# import shutil
# for p in [BASE_MODEL_DIR, LORA_DIR, ROBERTA_DIR]:
#     if p.exists():
#         shutil.rmtree(p)
#         print('Deleted:', p)
#         p.mkdir(parents=True, exist_ok=True)


In [ ]:
!pip install -q -U "huggingface_hub>=0.24.0" transformers peft accelerate "bitsandbytes>=0.46.1"
!pip install -q rouge-score bert-score nltk anthropic openai ipywidgets scipy matplotlib
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('Dependencies installed.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 123.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.9/923.9 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 105.5 MB/s eta 0:00:00
Dependencies installed.


In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = ""
def get_secret(name, default=None):
    value = os.environ.get(name)
    if value:
        return value
    try:
        return userdata.get(name) or default
    except Exception:
        return default
HF_TOKEN = get_secret('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Missing HF_TOKEN. Add it in Colab Secrets first.')
print('HF token found.')


HF token found.


In [ ]:
from huggingface_hub import snapshot_download

def download_if_missing(repo_id, local_dir, required_file, token=None, ignore_patterns=None):
    marker = Path(local_dir) / required_file
    if marker.exists():
        print('Already present, skipping:', local_dir)
        return
    print(f'Downloading {repo_id} -> {local_dir}')
    snapshot_download(repo_id=repo_id, local_dir=str(local_dir), token=token, ignore_patterns=ignore_patterns)
    if not marker.exists():
        raise RuntimeError(f'Missing expected file after download: {marker}')
    print('Done:', repo_id)

download_if_missing('Qwen/Qwen2.5-32B-Instruct', BASE_MODEL_DIR, 'config.json', token=HF_TOKEN, ignore_patterns=['*.bin', 'original/*'])
download_if_missing('lukelu33/cbt-qwen32b-lora', LORA_DIR, 'adapter_config.json', token=HF_TOKEN)
download_if_missing('FacebookAI/roberta-large', ROBERTA_DIR, 'config.json', token=HF_TOKEN, ignore_patterns=['*.bin'])


Already present, skipping: /content/drive/MyDrive/models/Qwen2.5-32B-Instruct


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Done: lukelu33/cbt-qwen32b-lora


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Done: FacebookAI/roberta-large


In [ ]:
import json
paths = {'base_model_dir': str(BASE_MODEL_DIR), 'lora_dir': str(LORA_DIR), 'roberta_dir': str(ROBERTA_DIR), 'result_root': str(RESULT_ROOT)}
with open(RESULT_ROOT / 'colab_paths.json', 'w', encoding='utf-8') as f:
    json.dump(paths, f, ensure_ascii=False, indent=2)
print('Asset preparation complete.')
print(json.dumps(paths, indent=2))


Asset preparation complete.
{
  "base_model_dir": "/content/drive/MyDrive/models/Qwen2.5-32B-Instruct",
  "lora_dir": "/content/drive/MyDrive/models/cbt-qwen32b-lora",
  "roberta_dir": "/content/drive/MyDrive/models/roberta-large",
  "result_root": "/content/drive/MyDrive/cbt_results"
}


In [ ]:
#import shutil
#shutil.rmtree("/content/drive/MyDrive/models/Qwen2.5-32B-Instruct")

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

BASE_MODEL_DIR = Path("/content/drive/MyDrive/models/Qwen2.5-32B-Instruct")

snapshot_download(
    repo_id="Qwen/Qwen2.5-32B-Instruct",
    local_dir=str(BASE_MODEL_DIR),
    token=HF_TOKEN,
    resume_download=True,
    local_dir_use_symlinks=False,
    allow_patterns=[
        "*.json",
        "*.safetensors",
        "*.model",
        "*.txt",
        "*.md",
        "LICENSE",
        ".gitattributes",
    ],
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 27 files:   0%|          | 0/27 [00:00<?, ?it/s]

'/content/drive/MyDrive/models/Qwen2.5-32B-Instruct'

In [ ]:
from pathlib import Path

base = Path("/content/drive/MyDrive/models/Qwen2.5-32B-Instruct")
safetensors = list(base.glob("*.safetensors"))

print("safetensors:", len(safetensors))
print("total GB:", sum(f.stat().st_size for f in safetensors) / 1024**3)
print([f.name for f in safetensors[:5]])
print("index exists:", (base / "model.safetensors.index.json").exists())

safetensors: 17
total GB: 61.027558386325836
['model-00001-of-00017.safetensors', 'model-00002-of-00017.safetensors', 'model-00003-of-00017.safetensors', 'model-00004-of-00017.safetensors', 'model-00005-of-00017.safetensors']
index exists: True


In [ ]:
from pathlib import Path

lora = Path("/content/drive/MyDrive/models/cbt-qwen32b-lora")

print("adapter_config:", (lora / "adapter_config.json").exists())
print("adapter_model:", (lora / "adapter_model.safetensors").exists())
print("tokenizer:", (lora / "tokenizer.json").exists())
print("files:", [p.name for p in lora.glob("*")])
print("adapter GB:", (lora / "adapter_model.safetensors").stat().st_size / 1024**3)

adapter_config: True
adapter_model: True
tokenizer: True
files: ['.cache', 'adapter_config.json', 'adapter_model.safetensors', 'special_tokens_map.json', 'merges.txt', 'tokenizer_config.json', '.gitattributes', 'tokenizer.json', 'added_tokens.json']
adapter GB: 2.0001139119267464
